In [0]:
%sql
used table: tblDepartment10, tblEmployee10

Derived tables and common table expressions (CTE's). We will also explore the differences between Views, Table Variable, Local and Global Temp Tables, Derived tables and common table expressions.
                                             
-- Now, we want to write a query which would return the following output. The query should return, the Department Name and Total Number of employees, with in the department. The departments with greatar than or equal to 2 employee should only be returned.  
+----------+----------------+
| DeptName | TotalEmployees |
+----------+----------------+
| IT       | 2              |
| HR       | 2              |
+----------+----------------+

select * from tblDepartment10
-- DeptId	DeptName
-- 1	IT
-- 2	Payroll
-- 3	HR
-- 4	Admin

select * from tblEmployee10
EmpId	EmpName	Gender	DepartmentId
1	John	Male	3
2	Mike	Male	2
3	Pam	Female	1
4	Todd	Male	4
5	Sara	Female	1
6	Ben	Male	3

-- 1 Obviously, there are severl ways to do this. Let's see how to achieve this, with the help of a view
-- Script to create the View
alter view vWEmployeeCount 
as 
select DeptName, DeptId, count(*) as TotalEmployees from tblDepartment10
join tblEmployee10
on tblDepartment10.DeptId = tblEmployee10.DepartmentId
Group By  DeptName, DeptId

select * from vWEmployeeCount

-- Query using the view:
Select DeptName, TotalEmployees 
from vWEmployeeCount
where  TotalEmployees >= 2

Note: Views get saved in the database, and can be available to other queries and stored procedures. However, if this view is only used at this one place, it can be easily eliminated using other options, like CTE, Derived Tables, Temp Tables, Table Variable etc.

-- 2 Now, let's see, how to achieve the same using, temporary tables(#TempEmployeeCount). We are using local temporary tables here.
Select DeptName, DepartmentId, COUNT(*) as TotalEmployees
into #TempEmployeeCount
from tblEmployee10
join tblDepartment10
on tblEmployee10.DepartmentId = tblDepartment10.DeptId
group by DeptName, DepartmentId

Select DeptName, TotalEmployees
From #TempEmployeeCount
where TotalEmployees >= 2

Drop Table #TempEmployeeCount

-- Note: Temporary tables are stored in TempDB. Local temporary tables are visible only in the current session, and can be shared between nested stored procedure calls. Global temporary tables are visible to other sessions and are destroyed, when the last connection referencing the table is closed.

-- Using Table Variable:
Declare @tblEmployeeCount table
(DeptName nvarchar(20),DepartmentId int, TotalEmployees int)

Insert @tblEmployeeCount
Select DeptName, DepartmentId, COUNT(*) as TotalEmployees
from tblEmployee
join tblDepartment
on tblEmployee.DepartmentId = tblDepartment.DeptId
group by DeptName, DepartmentId

Select DeptName, TotalEmployees
From @tblEmployeeCount
where  TotalEmployees >= 2

Note: Just like TempTables, a table variable is also created in TempDB. The scope of a table variable is the batch, stored procedure, or statement block in which it is declared. They can be passed as parameters between procedures.

-- Using Derived Tables
Select DeptName, TotalEmployees
from 
 (
  Select DeptName, DepartmentId, COUNT(*) as TotalEmployees
  from tblEmployee10
  join tblDepartment10
  on tblEmployee10.DepartmentId = tblDepartment10.DeptId
  group by DeptName, DepartmentId
 ) 
as EmployeeCount -- optional 
where TotalEmployees >= 2

Note: Derived tables are available only in the context of the current query.

-- Using CTE
With EmployeeCount(DeptName, DepartmentId, TotalEmployees)
  --  OR
With EmployeeCount
as
(
 Select DeptName, DepartmentId, COUNT(*) as TotalEmployees
 from tblEmployee10
 join tblDepartment10
 on tblEmployee10.DepartmentId = tblDepartment10.DeptId
 group by DeptName, DepartmentId
)
Select DeptName, TotalEmployees
from EmployeeCount
where TotalEmployees >= 2

Note: A CTE can be thought of as a temporary result set that is defined within the execution scope of a single SELECT, INSERT, UPDATE, DELETE, or CREATE VIEW statement. A CTE is similar to a derived table in that it is not stored as an object and lasts only for the duration of the query.

